# Explore to clean columns

in obt.py:
- event dates str ->  to_timestamp
- Event distance/length -> find the extremes, exclude? + supress rows with Etappen
- athlete avg spped -> find the extreme values, exclude them vs google search average + rounding if needed
- "remove all invalid rows (for simplicity you can consider d (days) as invalid and remove it)" 
- "convert time of athelete performance to appropriate data type"
- "Create relevant ids, which will be important later on dimensional modeling."
- delete the * in athlete club
- new column with the age at the time of the event + eclude the impossible
- after each clenaing, mentin the number of rows left?

In [0]:
df = spark.sql("SELECT * FROM marathos_cat.bronze.raw_marathos")
df.limit(3).display()

## All in snake_case

In [0]:
# --- Testing the function in utils here --- 
# copied the function -
import re

def to_snake_case(name):
    name = re.sub(r"\s*/\s*", "_or_", name.strip().casefold())
    name = re.sub(r"\s+", "_", name)
    return name

def rename_columns_to_snake_case(df):
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns)

# testing it here
df = rename_columns_to_snake_case(df)
df.printSchema()

## Converting column "event_dates" from str to to_date

In [0]:
df_cleaned = rename_columns_to_snake_case(df)


In [0]:
#from pyspark.sql.functions import col, to_date

# --- to_date ---
#df_cleaned = df_cleaned.withColumn("event_dates", to_date(col("event_dates"), "dd.MM.yyyy"))
from pyspark.sql.functions import try_to_date

df_cleaned = df_cleaned.withColumn("event_dates", try_to_date(col("event_dates"), "dd.MM.yyyy"))
# testing it
df_cleaned.printSchema()

## Column event_distance_or_length AND athlete_performance

In [0]:
'''Here I am first removing the elements discussed in the classroom like values containing "d" for "days", or "Etappen".'''


# --- First counting values containing "d" for days ---
# Used LLM for the regex part
d_count = df_cleaned.filter(col("event_distance_or_length").rlike(r"^\d+\.?\d*d")).count()
#print(d_count)

# --- Then counting values containing "Etappen" --- 
etappen_count = df_cleaned.filter(col("event_distance_or_length").contains("Etappen")).count()
#print(etappen_count)

total = df_cleaned.count()
#print(total)

print(f"Total rows: {total}")
print(f"Rows with d: {d_count}")
print(f"Rows with Etappen: {etappen_count}")
print(f"Rows remaining after removal: {total - d_count - etappen_count}")

# -- Now actually removing them by "ignoring" them with ~ ---
df_cleaned = df_cleaned.filter(~col("event_distance_or_length").rlike(r"^\d+\.?\d*d"))
df_cleaned = df_cleaned.filter(~col("event_distance_or_length").contains("Etappen"))

# --- check --- 
print(f"Rows after removal: {df_cleaned.count()}")

In [0]:
'''Here I separate the numbers from the letters or units of measurement like km, miles etc.. to supress or rename and regroup them eventually'''

from pyspark.sql.functions import regexp_extract

df_cleaned = df_cleaned.withColumn(
    "unit_measure",
    regexp_extract(col("event_distance_or_length"), r"[a-zA-Z]+", 0)
)

df_cleaned.groupBy("unit_measure").count().orderBy("count", ascending=False).show()

for km only:
- before 6069787+1931+159+52 = 6071929
- after 6071929


In [0]:
# --- Suppressing the dubitable ones ---
df_cleaned = df_cleaned.filter(~col("unit_measure").isin(["None", "m", "x", ""]))

# Checking
df_cleaned.count()

In [0]:
# --- Standardization of the relevant ones ---
from pyspark.sql.functions import when

df_cleaned = df_cleaned.withColumn(
    "unit_measure",
    when(col("unit_measure").isin(["km", "Km", "k", "K"]), "km")
    .when(col("unit_measure").isin(["mi", "Miles", "miles", "Mile", "mile"]), "mi")
    .when(col("unit_measure") == "h", "h")
    .otherwise(col("unit_measure"))
)

# checking
df_cleaned.groupBy("unit_measure").count().orderBy("count", ascending=False).show()

In [0]:
'''Above I have extracetd the unit measure, here I will extract the numric value and convert "miles" to "km" '''

# --- Extraction of the numeric part --- 
from pyspark.sql.functions import round as spark_round

# Below Regex r"(\d+\.?\d*)", 1) was produced by LLM
df_cleaned = df_cleaned.withColumn(
    "unit_value", 
    spark_round(
        when(col("unit_measure") == "mi", regexp_extract(col("event_distance_or_length"), r"(\d+\.?\d*)", 1)
        .cast("double") * 1.60934)
        .otherwise(regexp_extract(col("event_distance_or_length"), r"(\d+\.?\d*)", 1).cast("double")),
        2
    )
).withColumn(
    "unit_measure",
    when(col("unit_measure") == "mi", "km").otherwise(col("unit_measure"))
)

df_cleaned.select("unit_value", "unit_measure").show(10)

In [0]:
# --- Need to check df_cleaned so far ---
df_cleaned.limit(10).display()
## date error here, back to cell 89 from to_date to try_date? Many more nulls? Or can I just excludee range dates? or extract the irst date?

In [0]:
# remove the data comtaining "stage"?
# trying my luckk
df_cleaned.select("athlete_performance").distinct().show(50)


- integer for calculation
- str if not 
- casting = converting the type of a variable to another one
- convert into better type: "Athlete performance"
- need to round Athlete average speed till 2 decimals, spark_round
- clean "Athlete year of birth" vs event date lol
- .trim() and lower()
- if gender is not f or m = suppress or convert?

## Average speed

In [0]:
df_cleaned.select("athlete_average_speed").summary().show()

In [0]:
'''As per research online the "absolute maximum average speed for a marathon runner is approximately 21.1 km/h or  13.1mph which corresponds to the official marathon world record of 2h 35s. I will exclude unreasonable results'''

# --- Viewing the average speeds ---
df_cleaned.select("athlete_average_speed") \
    .distinct() \
    .orderBy("athlete_average_speed", ascending=False) \
    .display()

# --- datatype --- 
df_cleaned.select("athlete_average_speed").printSchema()

# Conversion float wold work but double is the PySpark default and rounding to 1 decimal ---
df_cleaned = df_cleaned.withColumn(
    "athlete_average_speed",
    spark_round(col("athlete_average_speed").cast("double"), 1)
)

# Step 2 — filter
df_cleaned = df_cleaned.filter(
    (col("athlete_average_speed") > 0) & (col("athlete_average_speed") <= 21)
)

# --- Excluding extremes ---

'''above note: some values have 18:00:00
try_cast like try_date will return nulls for invalid values instead of crashing.The 18:00:00 values will be null and the filter > 0 will remove them automatically since null > 0 is false.'''
